# Exploratory Data Analysis

In [30]:
import pandas as pd

In [31]:
from ast import literal_eval

In [32]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


## Data Retrieval

In [33]:
path = "/content/gdrive/My Drive/Colab Notebooks/Final Capstone Project /resume_clean.csv"

resume_df = pd.read_csv(path)

resume_df.head()

,ID,Resume_str,Resume_html,Category,clean_description,degrees
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr administrator marketing associate hr admini...,"['Associate', 'High School Diploma']"
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR,hr specialist us hr operations summary versati...,"['Master', 'Bachelor']"
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr director summary over 20 years experience i...,"['Master', 'Bachelor']"
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr specialist summary dedicated driven and dyn...,[]
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr manager skill highlights hr skills hr depar...,"['Associate', 'Bachelor']"


In [34]:
path = "/content/gdrive/My Drive/Colab Notebooks/Final Capstone Project /job_clean.csv"

job_df = pd.read_csv(path)

job_df.head()

,company_name,job_description,position_title,description_length,model_response,clean_description,Core Responsibilities,Required Skills,Educational Requirements,Experience Level,Preferred Qualifications,Compensation and Benefits,medical specialty,schedule,license/certification,degrees,required_skills_clean,skill_candidates
0,Google,minimum qualifications\nbachelors degree or eq...,Sales Specialist,2727,"{\n ""Core Responsibilities"": ""Responsible fo...",minimum qualifications bachelors degree or equ...,Responsible for expanding Google Workspace pro...,Bachelor's degree or equivalent experience. Ex...,Bachelor's degree or equivalent experience.,Experience managing enterprise SaaS accounts a...,Experience building strategic partnerships wit...,NaN,NaN,NaN,NaN,['Bachelor'],bachelor's degree or equivalent experience. ex...,"[""bachelor 's degree"", 'equivalent experience'..."
1,Apple,description\nas an asc you will be highly infl...,Apple Solutions Consultant,828,"{\n ""Core Responsibilities"": ""as an asc you ...",description as an asc you will be highly influ...,as an asc you will be highly influential in gr...,a passion to help people understand how apple ...,NaN,years preferred,NaN,NaN,NaN,NaN,NaN,[],a passion to help people understand how apple ...,"['apple product', 'livesexcellent communicatio..."
2,Netflix,its an amazing time to be joining netflix as w...,Licensing Coordinator - Consumer Products,3205,"{\n ""Core Responsibilities"": ""Help drive bus...",its an amazing time to be joining netflix as w...,Help drive business by supporting licensing ma...,2+ years experience in preferably outbound lic...,NaN,2+ years experience in preferably outbound lic...,Understanding of category manufacturing and sa...,NaN,NaN,NaN,NaN,[],2+ years experience in preferably outbound lic...,"['2 + year experience', 'preferably outbound l..."
3,Robert Half,description\n\nweb designers looking to expand...,Web Designer,2489,"{\n ""Core Responsibilities"": ""Designing webs...",description web designers looking to expand yo...,"Designing websites, wireframes, landing pages,...",2+ years experience in web design. Proficiency...,NaN,2+ years,UX/UI design experience. Knowledge of brand st...,NaN,NaN,NaN,NaN,[],2+ years experience in web design. proficiency...,"['2 + year experience', 'web design', 'strong ..."
4,TrackFive,at trackfive weve got big goals were on a miss...,Web Developer,3167,"{\n ""Core Responsibilities"": ""Build and layo...",at trackfive weve got big goals were on a miss...,"Build and layouts from provided PSD files, bui...","2+ years of experience with HTML and CSS/SASS,...",NaN,2+ years,NaN,Free health insurance for employees with no wa...,NaN,NaN,NaN,[],2+ years of experience with html and css sass ...,"['2 + year', 'programming php application', 'l..."


## Resume Datatset EDA

### Top Resume Categories

In [35]:
import plotly.express as px

category_counts = (
    resume_df["Category"]
    .value_counts()
    .reset_index()
)

category_counts.columns = ["Category", "Count"]

fig = px.bar(
    category_counts,
    x="Category",
    y="Count",
    title="Resume Categories",
    labels={
        "Category": "Category",
        "Count": "Count"
    }
)

fig.update_xaxes(tickangle=45)

fig.update_layout(
    width=1200,
    height=500
)

fig.show()

### Resume Length Distribution

In [44]:
resume_df["word_count"] = resume_df["clean_description"].str.split().str.len()

fig = px.histogram(
    resume_df,
    x="word_count",
    nbins=50,
    title="Distribution of Resume Description Word Counts",
    labels={
        "word_count": "Word Count",
        "count": "Number of Resume"
    }
)

fig.update_layout(
    width=900,
    height=500,
    bargap=0.05
)

fig.show()

### Most Common Words in Resumes

In [100]:
vectorizer = CountVectorizer(stop_words='english')

X = vectorizer.fit_transform(resume_df["Resume_str"])

word_counts = X.sum(axis=0).A1

vocab = vectorizer.get_feature_names_out()

common = (
    pd.DataFrame({
        "word": vocab,
        "count": word_counts
    })
    .sort_values("count", ascending=False)
)

In [101]:
top_words = common.head(20).sort_values("count")

fig = px.bar(
    top_words,
    x="count",
    y="word",
    orientation="h",
    title="Top 20 Most Frequent Words in Resume's",
    labels={
        "count": "Frequency",
        "word": "Word"
    },
    text="count"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=900,
    height=600,
    yaxis=dict(categoryorder="total ascending")
)

fig.show()

### Resume Bigrams

In [51]:
text = resume_df["clean_description"].dropna().astype(str)

# bigram
vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(2, 2),
    min_df=2
)

X = vectorizer.fit_transform(text)

bigram_counts = X.sum(axis=0).A1
bigrams = vectorizer.get_feature_names_out()

common_bigrams = (
    pd.DataFrame({
        "bigram": bigrams,
        "count": bigram_counts
    })
    .sort_values("count", ascending=False)
    .head(20)
)

fig = px.bar(
    common_bigrams.sort_values("count"),
    x="count",
    y="bigram",
    orientation="h",
    text="count",
    title="Top 20 Most Common Bigrams in Resume Descriptions",
    labels={
        "count": "Frequency",
        "bigram": "Bigram"
    }
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=900,
    height=700,
    yaxis=dict(categoryorder="total ascending")
)

fig.show()

### Resume Trigrams

In [53]:
text = resume_df["clean_description"].dropna().astype(str)

# bigram
vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(3, 3),
    min_df=3
)

X = vectorizer.fit_transform(text)

bigram_counts = X.sum(axis=0).A1
bigrams = vectorizer.get_feature_names_out()

common_bigrams = (
    pd.DataFrame({
        "bigram": bigrams,
        "count": bigram_counts
    })
    .sort_values("count", ascending=False)
    .head(20)
)

fig = px.bar(
    common_bigrams.sort_values("count"),
    x="count",
    y="bigram",
    orientation="h",
    text="count",
    title="Top 20 Most Common Trigrams in Resume Descriptions",
    labels={
        "count": "Frequency",
        "bigram": "Trigram"
    }
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=900,
    height=700,
    yaxis=dict(categoryorder="total ascending")
)

fig.show()

### Most Common Degrees in Resumes

In [61]:
# turn back to list
resume_df["degrees"] = resume_df["degrees"].apply(literal_eval)

degree_counts = (
    resume_df["degrees"]
    .explode()
    .dropna()
    .value_counts()
    .reset_index()
)

degree_counts.columns = ["Degree", "Count"]

fig = px.bar(
    degree_counts,
    x="Degree",
    y="Count",
    text="Count",
    title="Most Common Degrees in Resumes"
)

fig.update_layout(
    width=900,
    height=500,
    xaxis_title="Degree",
    yaxis_title="Number of Resumes"
)

fig.show()

## Job Posting EDA

### Top Job Position Titles

In [41]:
position_counts = (
    job_df["position_title"]
    .value_counts()
    .head(20)
    .sort_values()
    .reset_index()
)

position_counts.columns = ["Position Title", "Count"]

fig = px.bar(
    position_counts,
    x="Count",
    y="Position Title",
    orientation="h",
    title="Top 20 Most Common Position Titles",
    labels={
        "Position Title": "Position Title",
        "Count": "Number of Job Postings"
    }
)

fig.update_layout(
    width=1000,
    height=700
)

fig.show()

### Job Description Length

In [43]:
fig = px.histogram(
    job_df,
    x="description_length",
    nbins=50,
    title="Distribution of Job Description Word Counts",
    labels={
        "word_count": "Word Count",
        "count": "Number of Job Postings"
    }
)

fig.update_layout(
    width=900,
    height=500,
    bargap=0.05
)

fig.show()

### Most Common Words in Jobs

In [45]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words='english')

X = vectorizer.fit_transform(job_df["clean_description"])

word_counts = X.sum(axis=0).A1

vocab = vectorizer.get_feature_names_out()

common = (
    pd.DataFrame({
        "word": vocab,
        "count": word_counts
    })
    .sort_values("count", ascending=False)
)

In [46]:
top_words = common.head(20).sort_values("count")

fig = px.bar(
    top_words,
    x="count",
    y="word",
    orientation="h",
    title="Top 20 Most Frequent Words in Job Descriptions",
    labels={
        "count": "Frequency",
        "word": "Word"
    },
    text="count"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=900,
    height=600,
    yaxis=dict(categoryorder="total ascending")
)

fig.show()

### Job Bigrams

In [49]:
eeo_headers = [
    "equal opportunity employer",
    "equal employment opportunity",
    "we are an equal opportunity employer",
    "all qualified applicants"
]

def remove_eeo(text):

    lower = text.lower()

    for phrase in eeo_headers:
        idx = lower.find(phrase)

        if idx != -1:
            return text[:idx]

    return text

In [50]:
text = (
    job_df["clean_description"]
    .dropna()
    .astype(str)
    .apply(remove_eeo)
)

vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(2, 2),
    min_df=2
)

X = vectorizer.fit_transform(text)

bigram_counts = X.sum(axis=0).A1
bigrams = vectorizer.get_feature_names_out()

remove_bigrams = {
    "years experience",
    "team members",
    "ability work",
    "skills ability",
    "job description",
    "minimum years",
    "related field",
    "experience working",
    "new york",
    "policies procedures",
    "best practices",
    "internal external",
    "work environment"
}

common_bigrams = (
    pd.DataFrame({
        "bigram": bigrams,
        "count": bigram_counts
    })
    .query("bigram not in @remove_bigrams")
    .sort_values("count", ascending=False)
    .head(20)
)

fig = px.bar(
    common_bigrams.sort_values("count"),
    x="count",
    y="bigram",
    orientation="h",
    text="count",
    title="Top 20 Most Common Bigrams in Job Descriptions",
    labels={
        "count": "Frequency",
        "bigram": "Bigram"
    }
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=900,
    height=700,
    yaxis=dict(categoryorder="total ascending")
)

fig.show()

### Job Trigrams

In [52]:
text = job_df["clean_description"].dropna().astype(str)

# bigram
vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(3, 3),
    min_df=3
)

X = vectorizer.fit_transform(text)

bigram_counts = X.sum(axis=0).A1
bigrams = vectorizer.get_feature_names_out()

common_bigrams = (
    pd.DataFrame({
        "bigram": bigrams,
        "count": bigram_counts
    })
    .sort_values("count", ascending=False)
    .head(20)
)

fig = px.bar(
    common_bigrams.sort_values("count"),
    x="count",
    y="bigram",
    orientation="h",
    text="count",
    title="Top 20 Most Common Trigrams in Job Descriptions",
    labels={
        "count": "Frequency",
        "bigram": "Trigram"
    }
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=900,
    height=700,
    yaxis=dict(categoryorder="total ascending")
)

fig.show()

### Most Common Skills

In [55]:
job_df["skill_candidates"] = job_df["skill_candidates"].apply(literal_eval)

skills = (
    job_df["skill_candidates"]
    .explode()
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
)

skills = (
    skills
    .str.replace("’", "'", regex=False)
    .str.replace(r"\s*'\s*s\b", "'s", regex=True)
    .str.replace(r"\s*\+\s*", "+", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

remove = {
    "",
    "n a",
    "n/a",
    "required skill",
    "preferred skill",
    "qualification",
    "requirement",
    "experience",
    "candidate",
    "ability",
    "knowledge",
    "year experience",
    "fast pace environment",
    "work knowledge"
}

skills = skills[~skills.isin(remove)]

remove_pattern = (
    r"bachelor'?s?\s+degree"
    r"|high\s+school\s+diploma"
    r"|associate'?s?\s+degree"
    r"|master'?s?\s+degree"
    r"|related\s+field"
    r"|\b\d+\+?\s*years?(?:\s+experience)?\b"
    r"|driver'?s?\s+license"
    r"|customer\s+service\s+experience"
)

skills = skills[
    ~skills.str.contains(
        remove_pattern,
        case=False,
        regex=True,
        na=False
    )
]

skill_counts = (
    skills
    .value_counts()
    .head(15)
    .rename_axis("Skill")
    .reset_index(name="Count")
    .sort_values("Count", ascending=True)
)

skill_counts["Skill"] = skill_counts["Skill"].str.title()

fig = px.bar(
    skill_counts,
    x="Count",
    y="Skill",
    orientation="h",
    text="Count",
    title="Most Frequently Requested Skills",
    labels={
        "Count": "Number of Job Postings",
        "Skill": ""
    }
)

fig.update_layout(
    width=1000,
    height=650,
    xaxis_title="Number of Job Postings",
    yaxis_title="",
    yaxis=dict(categoryorder="total ascending"),
    showlegend=False
)

fig.show()

### Experience Requirement Distribution

In [66]:
job_df["Experience Level"].head(20)

,Experience Level
0,Experience managing enterprise SaaS accounts a...
1,years preferred
2,2+ years experience in preferably outbound lic...
3,2+ years
4,2+ years
5,NaN
6,2+ years experience as an individual contribut...
7,2+ years of relevant digital design experience
8,3+ years experience as a web designer
9,2+ years of experience


In [67]:
job_df["Experience Level"].value_counts(dropna=False).head(30)

,count
Experience Level,
NaN,266
5+ years,16
2+ years,13
3+ years,7
5 years,6
2 years,4
2+ years of experience,3
2+ years of relevant experience,3
2+ years of relevant work experience,2


In [80]:
#Extracting the minimum requirement of each experience level
import re
import numpy as np

def extract_experience_years(text):
    if pd.isna(text):
        return np.nan

    text = str(text).lower()

    #Remove age requirements
    text = re.sub(
        r'\b(?:at least\s+)?\d+\s+years?\s+(?:old|of age)\b',
        '',
        text)

    #Find numbers associated with year or years
    matches = re.findall(r'(\d+)\+?\s*(?:years?|yrs?)', text)

    if matches:
        #If multiple requirements appear, use the minimum
        return min(int(x) for x in matches)

    return np.nan

job_df["min_experience_years"] = (
    job_df["Experience Level"]
    .apply(extract_experience_years)
)

In [83]:
job_df[["position_title", "Experience Level", "min_experience_years"]].head(30)

,position_title,Experience Level,min_experience_years
0,Sales Specialist,Experience managing enterprise SaaS accounts a...,NaN
1,Apple Solutions Consultant,years preferred,NaN
2,Licensing Coordinator - Consumer Products,2+ years experience in preferably outbound lic...,2.0
3,Web Designer,2+ years,2.0
4,Web Developer,2+ years,2.0
5,Frontend Web Developer,NaN,NaN
6,Remote Website Designer,2+ years experience as an individual contribut...,2.0
7,Web Designer,2+ years of relevant digital design experience,2.0
8,Web Designer,3+ years experience as a web designer,3.0
9,SR. Web Designer,2+ years of experience,2.0


In [84]:
#Checking whether these extraction worked
job_df["min_experience_years"].value_counts().sort_index()

,count
min_experience_years,
0.0,1
1.0,54
2.0,212
3.0,51
4.0,8
5.0,135
6.0,1
7.0,4
8.0,3


In [85]:
print("Jobs with extracted experience:",job_df["min_experience_years"].notna().sum())
print("Jobs without numeric experience:",job_df["min_experience_years"].isna().sum())

Jobs with extracted experience: 480
Jobs without numeric experience: 373


In [93]:
experience_counts = (
    job_df["min_experience_years"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .reset_index()
)

experience_counts.columns = ["Years of Experience", "Number of Jobs"]

fig = px.bar(
    experience_counts,
    x="Years of Experience",
    y="Number of Jobs",
    text="Number of Jobs",
    title="Minimum Experience Requirements in Job Postings"
)

fig.update_layout(
    width=900,
    height=500,
    xaxis_title="Minimum Years of Experience",
    yaxis_title="Number of Job Postings"
)

fig.show()

Age statements such as "at least 18 years old" were initially detected as experience requirements. The extraction logic was refined to exclude age-related values.

### Education Requirements

In [95]:
job_df[["position_title", "Educational Requirements", "degrees"]].head(30)

,position_title,Educational Requirements,degrees
0,Sales Specialist,Bachelor's degree or equivalent experience.,[Bachelor]
1,Apple Solutions Consultant,NaN,[]
2,Licensing Coordinator - Consumer Products,NaN,[]
3,Web Designer,NaN,[]
4,Web Developer,NaN,[]
5,Frontend Web Developer,NaN,[]
6,Remote Website Designer,NaN,[]
7,Web Designer,NaN,[]
8,Web Designer,NaN,[]
9,SR. Web Designer,Bachelor's degree or equivalent experience.,[Bachelor]


In [96]:
#Measuring how much education inforamtion we have
total_jobs = len(job_df)
education_specified = job_df["Educational Requirements"].notna().sum()
education_missing = job_df["Educational Requirements"].isna().sum()

print("Total number of jobs:", total_jobs)
print("Number of jobs with education specified:", education_specified)
print("Number of jobs without education specified:", education_missing)

print("Percentage with education information:",round(education_specified / total_jobs * 100, 2),"%")

Total number of jobs: 853
Number of jobs with education specified: 461
Number of jobs without education specified: 392
Percentage with education information: 54.04 %


In [97]:
degree_missing = job_df[job_df["Educational Requirements"].notna()
    & job_df["degrees"].apply(lambda x: len(x) == 0)]

print("Education text present but no degree extracted:", len(degree_missing))

degree_missing[["position_title", "Educational Requirements", "degrees"]].head(20)

Education text present but no degree extracted: 117


,position_title,Educational Requirements,degrees
13,Wordpress Web Developer,"a bachelors degree in computer science, engine...",[]
28,"Performance Marketing Specialist, Paid Media",College degree required in relevant field of s...,[]
30,Software Engineer,"BSBA or greater in computer science, mathemati...",[]
31,Entry Level Software Engineer,College degree (associates or bachelors).,[]
36,Senior Software Engineer,"BS degree in computer science, engineering or ...",[]
37,Warehouse Manager,"High school education or equivalent, additiona...",[]
47,Part-time Bank Teller/Financial Services Repre...,High school degree or equivalent,[]
60,Account Executive (Inside Sales),College degree preferred but we have successfu...,[]
62,Full Stack Web Developer,bachelors degree in information systems softwa...,[]
73,"Analyst II, Supply Chain - Corporate - US (Open)",bachelors degree required,[]


###Degree Mentions in Job Posting

In [99]:
# turn back to list
#job_df["degrees"] = job_df["degrees"].apply(literal_eval)

degree_counts = (
    job_df["degrees"]
    .explode()
    .dropna()
    .value_counts()
    .reset_index()
)

degree_counts.columns = ["Degree", "Count"]

fig = px.bar(
    degree_counts,
    x="Degree",
    y="Count",
    text="Count",
    title="Most Common Degrees in Jobs"
)

fig.update_layout(
    width=900,
    height=500,
    xaxis_title="Degree",
    yaxis_title="Number of Mentions"
)

fig.show()

## Relationship Analysis

### Resume Length Vs Category

In [106]:
#Do resumes from different job categories tend to have different lengths?
resume_df["word_count"] = (resume_df["Resume_str"].fillna("").str.split().str.len())

top_categories = resume_df["Category"].value_counts().head(10).index
resume_top = resume_df[resume_df["Category"].isin(top_categories)]

fig = px.box(
    resume_top,
    x="Category",
    y="word_count",
    title="Resume Length Distribution by Category",
    labels={
        "Category": "Resume Category",
        "word_count": "Resume Word Count"
    }
)

fig.update_layout(
    width=1000,
    height=600,
    xaxis_tickangle=-45
)

fig.show()

Resume lengths are relatively similar across the major career categories, with most categories showing comparable median word counts. However, several outliers are present, representing unusually long or short resumes. This suggests that resume length varies more at the individual level than by career category.

### Experience Requirements Vs Job Description Length

In [108]:
#Do jobs requiring more experience tend to have longer job descriptions?

experience_jobs = job_df.dropna(subset=["min_experience_years", "description_length"])

fig = px.scatter(
    experience_jobs,
    x="min_experience_years",
    y="description_length",
    trendline="ols",
    title="Experience Requirements Vs Job Description Length",
    labels={
        "min_experience_years": "Minimum Years of Experience",
        "description_length": "Job Description Length"
    },
    opacity = 0.65
)

fig.update_layout(
    width=1000,
    height=600
)

fig.show()


In [109]:
correlation = experience_jobs[["min_experience_years", "description_length"]].corr().iloc[0, 1]
print("Correlation:", round(correlation, 3))

Correlation: 0.101


The correlation between minimum experience requirements and job description length is 0.101, indicating a very weak positive relationship. Although jobs requiring more experience may have slightly longer descriptions, experience level does not appear to be a strong factor in determining job description length. The wide variation in description lengths suggests that other factors contribute more to how detailed a job posting is.

### TF-IDF + PCA Visualization

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
import plotly.express as px

#use the cleaned resume text
resume_text = resume_df["clean_description"].fillna("")

#convert resume text into TF-IDF vectors
vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(resume_text)

print(X.shape)


(2484, 40156)


In [38]:
#Reduce TF-IDF vectors to dimensions
pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X.toarray())
print(X_reduced.shape)

(2484, 2)


In [39]:
plot_df = pd.DataFrame({
    "x": X_reduced[:, 0],
    "y": X_reduced[:, 1],
    "category": resume_df["Category"]
})
plot_df.head()

,x,y,category
0,0.062819,0.128927,HR
1,-0.024098,0.048465,HR
2,0.049015,-0.022393,HR
3,0.020231,0.102980,HR
4,0.087619,-0.012860,HR


In [40]:

#Scatter plot
fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    color="category",
    title="Resume Categories based on Text Similarity",
    opacity=0.65
)

fig.update_traces(marker=dict(size=6))

fig.update_layout(
    width=1000,
    height=700,
    legend_title = "Resume Category"
)

fig.show()

PCA was used to visualize high-dimensional TF-IDF resume features in two dimensions. The visualization showed considerable overlap between resume categories, suggesting that resumes from different career fields often share similar vocabulary and textual patterns.